In [137]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [138]:
import seaborn as sns

df = sns.load_dataset("penguins")
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [139]:
df = df.dropna()

In [140]:
y = df["species"]
X = df.drop(columns=["species"])

In [141]:
print(df["species"].unique())


['Adelie' 'Chinstrap' 'Gentoo']


# Creating The Model

In [142]:

def getsplit(X, column):
    # we will make a list that contains every possible split for this column 
    values = X[column].unique()
    if values.dtype == "object":
        return values.tolist()
    else:
        values = sorted(values)
        split = []
        for i in range(len(values)-1):
            avg = (values[i]+values[i+1])/2
            split.append(round(avg, 2))
        return split


In [143]:
def split_data(df, column, value):
    # if it is an object we will split the data : the left will take the lines where col == value and right will take the rest
    if df[column].dtype == "object":
        left = df[df[column] == value]
        right = df[df[column] != value]
    # if it is an int/float we will split the data : the left will take the lines where col < value and right will take the rest
    else:
        left = df[df[column] < value]
        right = df[df[column] >= value]
    return (left,right)

In [144]:
def mse(data, target):
    y = data[target]
    y_mean = y.mean()
    error = 0
    for y_i in y:
        error += (y_i - y_mean)**2
    return error *(1 / len(y))

In [145]:
def weighted_mse(left, right, target):
    nb_left, nb_right = len(left), len(right)
    total = nb_left + nb_right
    total_mse = (nb_left / total) * mse(left, target) + (nb_right / total) * mse(right, target)
    return total_mse

In [146]:
def best_split(df, target):
    # we intialize the variables
    best_wg, best_split, best_column = float("inf"), None, None
    X = df.drop(target, axis = 1)
    for column in X.columns:
        # for every column we will get a list of the possible splits
        splits = getsplit(X, column)
        for split in splits:
            # for every split we will split it again so that it would be left, right respecting the rules that we mentionned before
            left, right = split_data(df, column, split)
            if len(left) == 0 or len(right) == 0:
                continue
            wg = weighted_mse(left, right, target)
            # choosing the best split (the one that has the least wg)
            if wg < best_wg :
                best_wg = wg
                best_split = split
                best_column = column
    return best_wg, best_split, best_column

In [147]:
def should_stop(df, target, depth, max_depth, min_samples_split):
    if  df[target].nunique() == 1: # y contains only 1 unique value
        return True
    elif max_depth is not None and depth >= max_depth : # max depth reached
        return True
    elif len(df) < min_samples_split : # not enough samples
        return True
    return False

In [148]:
def create_leaf(df, target):
    # we create a leaf and the most appearing key will be the predicted value 
    prediction = df[target].mean()
    return {"type": "leaf" , "prediction" : prediction}
        

In [149]:
def build_tree(df, target, depth=0, max_depth=None, min_samples_split=2):
    # reccursive function that build a tree
    if should_stop(df, target, depth, max_depth, min_samples_split):
        return create_leaf(df, target)
    best_wg, best_value, best_column = best_split(df, target)
    if best_column is None:
        return create_leaf(df, target)
    left, right = split_data(df, best_column, best_value)
    left_tree = build_tree(left, target, depth+1, max_depth, min_samples_split)
    right_tree = build_tree(right, target, depth+1, max_depth, min_samples_split)
    return {
    "type": "node",
    "column": best_column,
    "value": best_value,
    "left": left_tree,
    "right": right_tree
    }

# Gradient Boosting

In [ ]:
def initialize_model(y):
    initial_prediction = np.mean(y)
    predictions = np.full(len(y), initial_prediction)
    return predictions

In [ ]:
def pseudo_residual(y, prediction):
    return y - prediction

In [ ]:
def (df, y, m, learning_rate, max_depth):
# df is d without y
    predictions = initialize_model(y)
    trees = []
    for _ in range(m):
        df["residual"] = pseudo_residual(y, predictions)
        tree = build_tree(df, "residual", max_depth)
        residual_predictions = predict_multiple_lines(tree, X)
        predictions += learning_rate * residual_predictions
        trees.append(tree)
    return initial_prediction, trees

In [150]:
def predict(tree, sample):
    if tree["type"] == "leaf" :
        return tree["prediction"]
    column = tree["column"]
    value = tree["value"]

    if isinstance(value, (int, float, np.number)): # we see if the value is numerical or not
        if sample[column] < value :
            return predict(tree["left"], sample) 
        else:
            return predict(tree["right"], sample)
    else:
        if sample[column] == value :
            return predict(tree["left"], sample) 
        else:
            return predict(tree["right"], sample)
        
    

In [151]:
def predict_multiple_lines(tree, X):
    predictions = []
    for _, row in X.iterrows():
        predictions.append(predict(tree, row))
    return predictions

In [152]:
def accuracy(y_true, y_pred):
    correct = 0

    for true, pred in zip(y_true, y_pred):
        if true == pred:
            correct += 1

    return correct / len(y_true)

# Predicting result

In [153]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

In [154]:
tree = build_tree(train, "species", max_depth=3)

In [155]:
X_test = test.drop("species", axis=1)
y_test_me = test["species"]

predictions = predict_multiple_lines(tree, X_test)

In [156]:
print(accuracy(y_test_me, predictions))

0.9850746268656716


# Creating the same model using sklearn

In [157]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X = df.drop("species", axis=1)
y = df["species"]
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X["island"] = encoder.fit_transform(X["island"])
X["sex"] = encoder.fit_transform(X["sex"])
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
model = DecisionTreeClassifier(
    max_depth=3
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# my model vs scikit-learn 

In [158]:
print(f" my accuracy :{accuracy(y_test_me, predictions)} vs scikit-learn's :{accuracy_score(y_test, y_pred)}")

 my accuracy :0.9850746268656716 vs scikit-learn's :0.9850746268656716


# Conclusion : it gives the same accuracy